<a href="https://colab.research.google.com/github/rayanban-bit/Bluetute/blob/main/p_T16_f_get_the_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#step 2 get the data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, datetime
PROJ = "/content/drive/MyDrive/t16_gaia"
os.makedirs(f"{PROJ}/data", exist_ok=True)
os.chdir(PROJ)
print("Working in", os.getcwd())

!pip install -q astroquery
import numpy as np, pandas as pd
from astropy.table import Table
from astroquery.vizier import Vizier
Vizier.ROW_LIMIT = -1                     # download whole catalogues, not 50 rows

LOG = []                                  # data provenance log (goes in the paper)
def log(name, n, source):
    LOG.append(dict(dataset=name, rows=n, source=source,
                    retrieved=datetime.date.today().isoformat()))
    print(f"{name:28s} {n:7d} rows   <- {source}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in /content/drive/MyDrive/t16_gaia


In [ ]:
# CELL 2 (final) - T16 candidate catalogue
from google.colab import files
up = files.upload()                       # select the Candidate catalog MRT file
name = list(up)[0]
os.replace(name, f"data/{name}")
t4 = Table.read(f"data/{name}", format="ascii.mrt").to_pandas()
t4.to_csv("data/t16_candidates.csv", index=False)
log("T16 candidate catalog", len(t4), f"ApJS 284, 19, MRT {name}")
print(len(t4)); print(list(t4.columns))

Saving t16_catalog_list_table.txt to t16_catalog_list_table.txt
T16 candidate catalog          11143 rows   <- ApJS 284, 19, MRT t16_catalog_list_table.txt
11143
['GAIAID', 'TICID', 'Sector', 'Cam', 'CCD', 'Teff', 'Lstar', 'Rstar', 'Mstar', 'Tmag', 'mh-gspphot', 'qtran', 'qingress', 'P', 'e_P', 'E_P', 'Tc', 'e_Tc', 'E_Tc', 'a/Rs', 'e_a/Rs', 'E_a/Rs', 'rp/rs', 'e_rp/rs', 'E_rp/rs', 'b', 'e_b', 'E_b', 'q1', 'e_q1', 'E_q1', 'q2', 'e_q2', 'E_q2', 'S/PN', 'Ntransits', 'RAdeg', 'DEdeg']


In [ ]:
# CELL 3 (while ExoFOP is down) - TOIs from the NASA Exoplanet Archive

import requests, io
url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
       "select+*+from+toi&format=csv")
r = requests.get(url, timeout=300)
r.raise_for_status()
toi = pd.read_csv(io.StringIO(r.text))
toi.to_csv("data/toi_nea.csv", index=False)
log("TOIs (NASA Exoplanet Archive)", len(toi), "NEA TAP toi table")
print(toi[["toi", "tid", "tfopwg_disp", "pl_orbper", "ra", "dec"]].head())
print(toi["tfopwg_disp"].value_counts())

TOIs (NASA Exoplanet Archive)    8151 rows   <- NEA TAP toi table
       toi        tid tfopwg_disp  pl_orbper          ra        dec
0  1017.01  182943944          FP   1.104220  124.726411 -37.783250
1  1018.01  291555748          FP   4.959120  111.716019 -50.266567
2  1019.01  341420329          PC   5.234101  120.147994 -54.878552
3   102.01  149603524          KP   4.411937   87.139821 -63.988329
4  1020.01  140706664          FP   3.288920  128.381976 -43.828421
tfopwg_disp
PC     4820
FP     1315
CP      815
KP      606
APC     490
FA      100
Name: count, dtype: int64


In [ ]:
# CELL 4 - eclipsing-binary catalogues the T16 team already used
# ---------------------------------------------------------------------
# Prsa et al. 2022, TESS EBs from 2-min data, sectors 1-26 (VizieR J/ApJS/258/16)
prsa = Vizier.get_catalogs("J/ApJS/258/16")
for t in prsa:
    print(t.meta.get("name"), len(t))
prsa_main = max(prsa, key=len).to_pandas()          # the largest table is the EB list
prsa_main.to_csv("data/prsa2022_ebs.csv", index=False)
log("Prsa+2022 TESS EBs", len(prsa_main), "VizieR J/ApJS/258/16")

# Kostov et al. 2025, TESS Ten Thousand EBs (FFI). Download Table 3 MRT from the
# ApJS supplement (or VizieR if listed) and upload as data/kostov2025_table3.txt
try:
    kos = Table.read("data/kostov2025_table3.txt", format="ascii.mrt").to_pandas()
    log("Kostov+2025 TESS 10k EBs", len(kos), "ApJS MRT")
except FileNotFoundError:
    print("Kostov table not uploaded yet")


J/ApJS/258/16/tess-ebs 4584
Prsa+2022 TESS EBs              4584 rows   <- VizieR J/ApJS/258/16
Kostov table not uploaded yet


In [ ]:
#kostov table (3) manually uploaded
kos_url = "https://content.cld.iop.org/journals/0067-0049/279/2/50/revision1/apjsade2d8t3_mrt.txt"
kos = Table.read(kos_url, format="ascii.mrt").to_pandas()
kos.to_csv("data/kostov2025_ebs.csv", index=False)
log("Kostov+2025 TESS 10k EBs", len(kos), "ApJS MRT " + kos_url.split("/")[-1])
print(len(kos)); print(list(kos.columns))

Kostov+2025 TESS 10k EBs        7936 rows   <- ApJS MRT apjsade2d8t3_mrt.txt
7936
['TIC', 'RAdeg', 'DEdeg', 'Tmag', 'Per', 'T0-pri', 'Depth-pri', 'Dur-pri', 'T0-sec', 'Phase', 'Depth-sec', 'Dur-sec', 'Sectors', 'RUWE', 'AEN', 'AENS', 'Teff', 'Com']


#step 3 Gaia DR3 match on T16 host stars

# Output: data/t16_gaia_hosts.csv, data/t16_host_flags.csv + summary counts.


In [ ]:
# CELL 1 - load the T16 catalogue and prepare the host list
import numpy as np, pandas as pd, time
from astroquery.gaia import Gaia
Gaia.ROW_LIMIT = -1

c = pd.read_csv("data/t16_candidates.csv")
c["GAIAID"] = pd.to_numeric(c["GAIAID"], errors="coerce").astype("Int64")
print("candidates:", len(c), "| missing Gaia ID:", c["GAIAID"].isna().sum())
print("unique host stars:", c["GAIAID"].nunique(),
      "| hosts with >1 candidate row:", (c["GAIAID"].value_counts() > 1).sum())

# Period uncertainty: T16 gives asymmetric errors (e_P lower, E_P upper); keep the larger
c["sigP"] = c[["e_P", "E_P"]].abs().max(axis=1)
c["sigTc"] = c[["e_Tc", "E_Tc"]].abs().max(axis=1)
c["dur"] = c["qtran"] * c["P"]                     # transit duration [d]
c["Tc_BJD"] = c["Tc"] + 2457000.0                  # T16 Tc is BJD - 2457000 (check the MRT header!)
ids = [int(x) for x in c["GAIAID"].dropna().unique()]


The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.
candidates: 11143 | missing Gaia ID: 0
unique host stars: 11143 | hosts with >1 candidate row: 0


In [ ]:
# CELL 2 - query Gaia DR3 in chunks (no table upload / login needed)
# ---------------------------------------------------------------------
Q_MAIN = """
SELECT g.source_id, g.ruwe, g.non_single_star, g.phot_g_mean_mag, g.phot_variable_flag,
       c.best_class_name, c.best_class_score,
       e.frequency, e.frequency_error, e.reference_time, e.model_type, e.global_ranking,
       e.derived_primary_ecl_phase, e.derived_primary_ecl_depth, e.derived_secondary_ecl_phase
FROM gaiadr3.gaia_source AS g
LEFT OUTER JOIN gaiadr3.vari_classifier_result AS c ON c.source_id = g.source_id
LEFT OUTER JOIN gaiadr3.vari_eclipsing_binary  AS e ON e.source_id = g.source_id
WHERE g.source_id IN ({})
"""
Q_NSS = """
SELECT source_id, nss_solution_type, period, period_error
FROM gaiadr3.nss_two_body_orbit WHERE source_id IN ({})
"""
def run_chunks(template, ids, size=800):
    out = []
    for i in range(0, len(ids), size):
        q = template.format(",".join(str(x) for x in ids[i:i + size]))
        for attempt in range(3):
            try:
                out.append(Gaia.launch_job_async(q).get_results().to_pandas()); break
            except Exception as err:
                print(f"  chunk {i // size}: {err} - retrying"); time.sleep(15)
        print(f"  done {min(i + size, len(ids))}/{len(ids)}")
    return pd.concat(out, ignore_index=True)

gaia = run_chunks(Q_MAIN, ids)
gaia.to_csv("data/t16_gaia_hosts.csv", index=False)       # save immediately
nss = run_chunks(Q_NSS, ids)
nss.to_csv("data/t16_gaia_nss.csv", index=False)
print("Gaia rows:", len(gaia), "| hosts with a Gaia EB entry:", gaia["frequency"].notna().sum(),
      "| NSS orbit solutions:", len(nss))


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 800/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 1600/11143


INFO:astroquery:Query finished.


...

INFO: Query finished. [astroquery.utils.tap.core]
  done 9600/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 10400/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 11143/11143
Gaia rows: 10927 | hosts with a Gaia EB entry: 28 | NSS orbit solutions: 132


In [ ]:
# Check completeness and re-query missing stars
got = set(gaia["source_id"].astype("int64"))
missing = [x for x in ids if x not in got]
print("missing from main query:", len(missing))
if missing:
    gaia = pd.concat([gaia, run_chunks(Q_MAIN, missing, size=300)], ignore_index=True)
    gaia = gaia.drop_duplicates(subset=["source_id"])
    gaia.to_csv("data/t16_gaia_hosts.csv", index=False)
    print("now missing:", len(set(ids) - set(gaia["source_id"].astype("int64"))))

missing from main query: 216


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 216/216
now missing: 216


In [ ]:
# Diagnose and recover the 216 missing hosts
miss = c[c["GAIAID"].isin(missing)].copy()
print(miss[["TICID", "GAIAID", "RAdeg", "DEdeg", "Tmag"]].head(10))
print("IDs ending in 000:", (miss["GAIAID"].astype(str).str[-3:] == "000").sum())  # sign of precision loss

# Test 1: are they Gaia DR2 IDs? Gaia provides a DR2 -> DR3 lookup table
Q_DR2 = """SELECT dr2_source_id, dr3_source_id, angular_distance, magnitude_difference
FROM gaiadr3.dr2_neighbourhood WHERE dr2_source_id IN ({})"""
dr2 = run_chunks(Q_DR2, [int(x) for x in missing], size=216)
print("found as DR2 IDs:", dr2["dr2_source_id"].nunique(), "of", len(missing))

# Keep the closest, best-matching DR3 counterpart for each DR2 ID
dr2 = (dr2.assign(score=dr2["angular_distance"] + 100 * dr2["magnitude_difference"].abs().fillna(9))
          .sort_values("score").drop_duplicates("dr2_source_id"))
id_map = dict(zip(dr2["dr2_source_id"], dr2["dr3_source_id"]))
c["GAIA_DR3"] = c["GAIAID"].map(lambda x: id_map.get(int(x), int(x)) if pd.notna(x) else x)
print("hosts re-mapped DR2 -> DR3:", (c["GAIA_DR3"] != c["GAIAID"]).sum())

         TICID               GAIAID  RAdeg  DEdeg      Tmag
27   389421473  6681850359495299968    NaN    NaN  12.61106
43   278536652  6573857740458966400    NaN    NaN  10.32008
101   92352620  6780546169633475456    NaN    NaN   9.64369
138  277683130  6409051044924482048    NaN    NaN   9.57036
139  406318573  6409854379902897536    NaN    NaN  13.26165
216  237885040  4705896508318372096    NaN    NaN  11.89497
367  284194284  5280996546265104896    NaN    NaN   9.69205
370  300164482  5281362992872721920    NaN    NaN  10.75675
373  177118691  5281812005936121088    NaN    NaN  10.94381
379  349095149  5282270089967734912    NaN    NaN  10.05784
IDs ending in 000: 2


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 216/216
found as DR2 IDs: 216 of 216
hosts re-mapped DR2 -> DR3: 216


In [ ]:
# Check the DR2 -> DR3 matches are unambiguous
print(dr2[["angular_distance", "magnitude_difference"]].describe())
print("matches > 1 arcsec apart:", (dr2["angular_distance"] > 1000).sum())   # angular_distance is in mas

# Query Gaia DR3 for the 216 remapped hosts and add them to the saved results
new_ids = [int(id_map[int(x)]) for x in missing]
g_new = run_chunks(Q_MAIN, new_ids, size=216)
gaia = pd.concat([gaia, g_new], ignore_index=True).drop_duplicates("source_id")
gaia.to_csv("data/t16_gaia_hosts.csv", index=False)
nss = pd.concat([nss, run_chunks(Q_NSS, new_ids, size=216)], ignore_index=True)
nss.to_csv("data/t16_gaia_nss.csv", index=False)
c.to_csv("data/t16_candidates.csv", index=False)       # saves the new GAIA_DR3 column
print("hosts now matched:", c["GAIA_DR3"].isin(gaia["source_id"]).sum(), "of", len(c))

       angular_distance  magnitude_difference
count        216.000000            216.000000
mean           1.850389              0.010838
std           10.326883              0.069302
min            0.009754             -0.018112
25%            0.056271             -0.013696
50%            0.099984             -0.010697
75%            0.208417              0.001660
max          120.241920              0.538751
matches > 1 arcsec apart: 0


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 216/216


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 216/216
hosts now matched: 11143 of 11143


In [ ]:
# CELL 3 - period and eclipse-time matching

m = c.merge(gaia, left_on="GAIA_DR3", right_on="source_id", how="left")

# Gaia EB period and its uncertainty
m["P_gaia"] = 1.0 / m["frequency"]
m["sigP_gaia"] = m["frequency_error"] / m["frequency"] ** 2
# Time of Gaia primary eclipse: reference_time is BJD(TCB) - 2455197.5; TCB-TDB (~20 s) is negligible
m["T_gaia"] = 2455197.5 + m["reference_time"] + m["derived_primary_ecl_phase"] / m["frequency"]

NSIG, FLOOR = 3.0, 1e-3            # 3-sigma window, plus a 0.1 % floor (Gaia errors can be optimistic)
m["k_match"] = np.nan
for k in (0.5, 1.0, 2.0):          # EBs are often detected at half or twice their true period
    Pk = k * m["P_gaia"]
    tol = np.maximum(NSIG * np.hypot(m["sigP"], k * m["sigP_gaia"]), FLOOR * m["P"])
    hit = (np.abs(m["P"] - Pk) < tol) & m["k_match"].isna()
    m.loc[hit, "k_match"] = k
m["period_match"] = m["k_match"].notna()

# Eclipse-time check: does a Gaia eclipse (primary or secondary) fall at the T16 transit time?
Puse = m["k_match"] * m["P_gaia"]
n_cyc = np.round((m["Tc_BJD"] - m["T_gaia"]) / Puse)
sigT = np.hypot(np.abs(n_cyc) * m["k_match"] * m["sigP_gaia"], m["sigTc"])
def offset(T0):                                  # days between T16 Tc and the nearest predicted eclipse
    x = (m["Tc_BJD"] - T0) / Puse
    return (x - np.round(x)) * Puse
off_p = offset(m["T_gaia"])
T_sec = m["T_gaia"] + (m["derived_secondary_ecl_phase"] - m["derived_primary_ecl_phase"]) / m["frequency"]
off_s = offset(T_sec)
m["ecl_offset_d"] = np.where(np.abs(off_s) < np.abs(off_p), off_s, off_p)
m["ecl_offset_d"] = np.where(m["derived_secondary_ecl_phase"].isna(), off_p, m["ecl_offset_d"])
m["epoch_unconstrained"] = m["period_match"] & (sigT > 0.25 * Puse)   # too many cycles to tell
m["epoch_match"] = (m["period_match"] & ~m["epoch_unconstrained"]
                    & (np.abs(m["ecl_offset_d"]) < NSIG * sigT + 0.5 * m["dur"]))

# Other Gaia binarity evidence
m["gaia_ecl_class"] = m["best_class_name"].eq("ECL")
m["ruwe_high"] = m["ruwe"] > 1.4
m["nss_any"] = m["non_single_star"].fillna(0) > 0
sb = nss[nss["nss_solution_type"].str.contains("SB1|SB2|EclipsingSpectro", na=False)]
m["nss_sb_orbit"] = m["GAIA_DR3"].isin(sb["source_id"])

m.to_csv("data/t16_host_flags.csv", index=False)


In [ ]:
# CELL 4 - summary for the report
n = len(m)
def frac(mask, label):
    k = int(mask.sum()); print(f"{label:55s} {k:6d}  ({100 * k / n:.2f} % of {n})")
frac(m["frequency"].notna(), "host in Gaia DR3 eclipsing-binary table")
frac(m["gaia_ecl_class"], "Gaia classifier best class = ECL")
frac(m["period_match"], "  Gaia EB period matches P, P/2 or 2P")
frac(m["epoch_match"], "  ... and eclipse time matches T16 transit time")
frac(m["epoch_unconstrained"], "  ... period match but eclipse time unconstrained")
frac(m["nss_sb_orbit"], "Gaia spectroscopic-binary orbit (SB1/SB2)")
frac(m["nss_any"], "any Gaia non-single-star solution")
frac(m["ruwe_high"], "RUWE > 1.4 (weaker evidence)")
print(m.loc[m["period_match"], "k_match"].value_counts())

# Inspect the strongest matches
cols = ["TICID", "GAIAID", "Tmag", "P", "P_gaia", "k_match", "ecl_offset_d",
        "epoch_match", "rp/rs", "derived_primary_ecl_depth", "ruwe", "model_type"]
m.loc[m["period_match"], cols].sort_values("epoch_match", ascending=False).head(25)


host in Gaia DR3 eclipsing-binary table                     28  (0.25 % of 11143)
Gaia classifier best class = ECL                            28  (0.25 % of 11143)
  Gaia EB period matches P, P/2 or 2P                        7  (0.06 % of 11143)
  ... and eclipse time matches T16 transit time              7  (0.06 % of 11143)
  ... period match but eclipse time unconstrained            0  (0.00 % of 11143)
Gaia spectroscopic-binary orbit (SB1/SB2)                   92  (0.83 % of 11143)
any Gaia non-single-star solution                          210  (1.88 % of 11143)
RUWE > 1.4 (weaker evidence)                              1031  (9.25 % of 11143)
k_match
1.0    6
2.0    1
Name: count, dtype: int64


,TICID,GAIAID,Tmag,P,P_gaia,k_match,ecl_offset_d,epoch_match,rp/rs,derived_primary_ecl_depth,ruwe,model_type
817,91527786,5066765467324661120,14.64508,4.587000,4.589050,1.0,-0.024818,True,0.1760,0.068199,0.998418,TWOGAUSSIANS
2904,429148337,3096727671604441856,15.24122,1.705850,0.852794,2.0,0.014361,True,0.1795,0.107422,1.051498,TWOGAUSSIANS
4541,47057078,3820592869736630528,13.82890,1.054252,1.054433,1.0,0.019494,True,0.2090,0.057768,1.101607,ONEGAUSSIAN
7312,290256766,5237178396595491456,15.97226,4.420700,4.411521,1.0,0.062068,True,0.1870,0.193321,1.769655,TWOGAUSSIANS_WITH_ELLIPSOIDAL_ON_ECLIPSE1
9753,406975511,5772263388139390592,15.40375,2.035100,2.035222,1.0,-0.001831,True,0.3590,0.231431,1.054444,TWOGAUSSIANS
9822,369366743,6346058138279996928,15.84578,0.782827,0.782637,1.0,-0.007137,True,0.2340,0.075230,1.411666,ONEGAUSSIAN
10558,292295736,6419427518410697856,15.35275,2.268510,2.269436,1.0,-0.049028,True,0.1910,0.077004,0.961232,TWOGAUSSIANS


In [ ]:
# 3b-1: spectroscopic-binary (SB1/SB2) orbit periods vs T16 periods
sbm = m[["TICID", "GAIA_DR3", "Tmag", "P", "sigP", "rp/rs"]].merge(
        sb, left_on="GAIA_DR3", right_on="source_id")
sbm["ratio"] = sbm["P"] / sbm["period"]
tol = np.maximum(3 * np.hypot(sbm["sigP"], sbm["period_error"].fillna(0)), 1e-3 * sbm["P"])
sbm["sb_match_k"] = np.nan
for k in (0.5, 1.0, 2.0):
    hit = (np.abs(sbm["P"] - k * sbm["period"]) < tol) & sbm["sb_match_k"].isna()
    sbm.loc[hit, "sb_match_k"] = k
print("SB orbits:", len(sbm), "| period-matched to T16:", sbm["sb_match_k"].notna().sum())
print(sbm.loc[sbm["sb_match_k"].notna(),
      ["TICID", "Tmag", "P", "period", "sb_match_k", "nss_solution_type", "rp/rs"]])
sbm.to_csv("data/t16_sb_matches.csv", index=False)

# 3b-2: the 21 Gaia eclipsing binaries without a period match: what period ratio?
um = m[(m["frequency"].notna()) & (~m["period_match"])].copy()
um["ratio"] = um["P"] / um["P_gaia"]
print(um[["TICID", "Tmag", "P", "P_gaia", "ratio", "derived_primary_ecl_depth", "model_type"]]
      .sort_values("ratio").to_string())

SB orbits: 92 | period-matched to T16: 55
        TICID      Tmag          P     period  sb_match_k nss_solution_type  \
0   278536652  10.32008   7.130200   7.131245         1.0               SB1   
1   393940766  11.75512   4.866300   4.873890         1.0               SB1   
2   182517223  10.36312   2.207590   2.207646         1.0               SB1   
6   100100827   8.82664   0.941373   0.941468         1.0               SB1   
7   300871545  10.03702   4.816560   4.817016         1.0               SB1   
9   439878401  10.23885   8.984000  17.967244         0.5               SB1   
10  168871868  11.42888   8.562700  17.117665         0.5               SB1   
12  153711481  12.02215  11.667100  11.666079         1.0               SB1   
13  260128333  11.48839   7.298800  14.629797         0.5               SB1   
14  169532369  11.57161   3.604070   3.604688         1.0               SB1   
15  140830390  11.71123  12.839600  12.851066         1.0               SB1   
16  167418

In [ ]:
# 3c: minimum companion mass from Gaia SB orbits + TOI disposition
Q_K = """SELECT source_id, nss_solution_type, period, semi_amplitude_primary,
                semi_amplitude_primary_error, eccentricity
         FROM gaiadr3.nss_two_body_orbit WHERE source_id IN ({})"""
kk = run_chunks(Q_K, [int(x) for x in sbm["GAIA_DR3"].unique()], size=100)
s = sbm.merge(kk[["source_id", "semi_amplitude_primary", "semi_amplitude_primary_error", "eccentricity"]],
              on="source_id", how="left").merge(c[["TICID", "Mstar"]], on="TICID", how="left")

# Mass function f = 1.0361e-7 (1-e^2)^1.5 K^3 P   [Msun; K in km/s, P in days]
e = s["eccentricity"].fillna(0)
f = 1.0361e-7 * (1 - e**2)**1.5 * s["semi_amplitude_primary"]**3 * s["period"]
M1 = s["Mstar"].fillna(1.0)                      # host mass [Msun] from T16 (Gaia-based)
M2 = (f * M1**2)**(1/3)                          # first guess, then iterate
for _ in range(20):
    M2 = (f * (M1 + M2)**2)**(1/3)
s["M2_MJup"] = M2 * 1047.6                       # transiting => sin i ~ 1, so this ~ true mass
s["class"] = pd.cut(s["M2_MJup"], [0, 13, 80, np.inf], labels=["planet", "brown dwarf", "star"])
s.loc[s["nss_solution_type"] == "SB2", "class"] = "star"

toi_disp = toi[["tid", "toi", "tfopwg_disp"]].rename(columns={"tid": "TICID"})
s = s.merge(toi_disp, on="TICID", how="left")
s.to_csv("data/t16_sb_masses.csv", index=False)

match = s[s["sb_match_k"].notna()]
print(pd.crosstab(match["class"], match["tfopwg_disp"].fillna("not a TOI"), margins=True))
print(match[["TICID", "Tmag", "P", "sb_match_k", "semi_amplitude_primary", "M2_MJup",
             "class", "toi", "tfopwg_disp"]].sort_values("M2_MJup").to_string())

INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 92/92
tfopwg_disp  APC  CP  FP  KP  PC  not a TOI  All
class                                           
planet         0   0   0   1   0          0    1
brown dwarf    0   0   0   0   1          0    1
star           6   2  12   0   3         31   54
All            6   2  12   1   4         31   56
        TICID      Tmag          P  sb_match_k  semi_amplitude_primary      M2_MJup        class      toi tfopwg_disp
6   100100827   8.82664   0.941373         1.0                1.938258    10.515269       planet   185.01          KP
22   52420398  11.65163   5.236900         1.0                7.864213    70.478270  brown dwarf  5427.01          PC
1   393940766  11.75512   4.866300         1.0                9.603169    91.006527         star   148.01          CP
74  457272901  11.61457   4.649780         1.0               11.736702    92.847937         star  3155.01         APC
18  167418903  11.36694  10.979600         1.0       

In [ ]:
# 3d-1: mass uncertainty by Monte Carlo (K error + assumed 10 % host-mass error)
rng = np.random.default_rng(1)
s1 = s.dropna(subset=["semi_amplitude_primary"]).drop_duplicates("TICID").copy()
N = 4000
def mc_mass(row):
    sigK = 0.0 if pd.isna(row.semi_amplitude_primary_error) else row.semi_amplitude_primary_error
    if pd.notna(row.Mstar):
        M1c, sM1 = row.Mstar, 0.10 * row.Mstar        # T16 (Gaia-based) mass, 10 % error
    else:
        M1c, sM1 = 1.0, 0.30                          # no mass given: assume 1.0 +/- 0.3 Msun
    K = rng.normal(row.semi_amplitude_primary, sigK, N).clip(0)
    M1 = rng.normal(M1c, sM1, N).clip(0.1)
    e = 0.0 if pd.isna(row.eccentricity) else row.eccentricity
    f = 1.0361e-7 * (1 - e**2)**1.5 * K**3 * row.period
    M2 = (f * M1**2)**(1/3)
    for _ in range(20):
        M2 = (f * (M1 + M2)**2)**(1/3)
    M2 *= 1047.6
    return pd.Series({"M2_med": np.median(M2), "M2_lo": np.percentile(M2, 16),
                      "M2_hi": np.percentile(M2, 84), "P_star": (M2 > 80).mean(),
                      "M1_assumed": pd.isna(row.Mstar)})
s1 = pd.concat([s1, s1.apply(mc_mass, axis=1)], axis=1)
s1["verdict"] = np.select([s1["nss_solution_type"].eq("SB2"), s1["P_star"] > 0.95,
                           s1["M2_hi"] < 13], ["star (SB2)", "star", "planet-mass"], "borderline")

# 3d-2: consolidated flag table (one row per T16 candidate)
eb = m[m["epoch_match"]][["TICID"]].assign(flag_EB_host=True)
eb = pd.concat([eb, pd.DataFrame({"TICID": [238074648], "flag_EB_host": [True]})])  # 3x alias
sbf = s1[s1["sb_match_k"].notna()][["TICID", "M2_med", "M2_lo", "M2_hi", "P_star", "verdict", "sb_match_k"]]
flags = c[["TICID", "GAIA_DR3", "Tmag", "P", "rp/rs"]].merge(eb, on="TICID", how="left") \
                                                    .merge(sbf, on="TICID", how="left")
flags = flags[flags["flag_EB_host"].fillna(False) | flags["verdict"].notna()]
flags = flags.merge(toi[["tid", "toi", "tfopwg_disp"]].rename(columns={"tid": "TICID"}), on="TICID", how="left")
flags.to_csv("data/t16_flags_hosts.csv", index=False)
print(flags["verdict"].value_counts(dropna=False))
print("total flagged host candidates:", len(flags))
print(flags.sort_values("M2_med").to_string())

verdict
star           49
NaN             8
borderline      3
star (SB2)      2
planet-mass     1
Name: count, dtype: int64
total flagged host candidates: 63
        TICID             GAIA_DR3      Tmag          P   rp/rs flag_EB_host       M2_med        M2_lo        M2_hi   P_star      verdict  sb_match_k      toi tfopwg_disp
3   100100827  4955371367334610048   8.82664   0.941373  0.0880          NaN    10.472578     9.465619    11.545835  0.00000  planet-mass         1.0   185.01          KP
15   52420398  3341401062125577088  11.65163   5.236900  0.0769          NaN    70.226101    60.056553    80.644080  0.17550   borderline         1.0  5427.01          PC
1   393940766  6594248591617970560  11.75512   4.866300  0.0537          NaN    90.258930    76.486236   105.208806  0.76650   borderline         1.0   148.01          CP
46  457272901  5869945139392446336  11.61457   4.649780  0.0951          NaN    92.664775    85.854473    99.605679  0.96700         star         1.0  3155.01

/tmp/ipykernel_3026/1968623831.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  flags = flags[flags["flag_EB_host"].fillna(False) | flags["verdict"].notna()]


# STEP 4 - Gaia DR3 eclipsing binaries NEAR each T16 host (blends)


###
A faint eclipsing binary inside the TESS aperture (21" pixels) can mimic
a transit on the host. We look for Gaia EBs within 60" (~3 TESS pixels),
then test (a) period + eclipse time and (b) whether the neighbour is
bright enough to produce the observed depth after dilution.
Run in the same notebook after Step 3 (needs c, gaia, run_chunks).
###



In [ ]:
# CELL 1 - host positions and G magnitudes (from Gaia DR3, all 11,143)
# ---------------------------------------------------------------------
import numpy as np, pandas as pd, time
from astropy.table import Table
from astroquery.gaia import Gaia

Q_POS = """SELECT source_id, ra, dec, phot_g_mean_mag
           FROM gaiadr3.gaia_source WHERE source_id IN ({})"""
pos = run_chunks(Q_POS, [int(x) for x in c["GAIA_DR3"].unique()])
hosts = c.merge(pos, left_on="GAIA_DR3", right_on="source_id", how="left")
print("hosts with Gaia positions:", hosts["ra"].notna().sum(), "of", len(hosts))
hosts[["TICID", "GAIA_DR3", "ra", "dec", "phot_g_mean_mag"]].to_csv("data/t16_host_pos.csv", index=False)


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 800/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 1600/11143


INFO:astroquery:Query finished.


...

INFO: Query finished. [astroquery.utils.tap.core]
  done 9600/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 10400/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 11143/11143
hosts with Gaia positions: 11143 of 11143


In [ ]:
# CELL 2 - spatial cross-match: all Gaia EBs within 60" of each host
# Uploads host positions to the Gaia archive in batches of 500.
# If anonymous upload is refused, create a free ESA Cosmos account and run
# Gaia.login(user="...", password="...") first.

RADIUS_DEG = 60 / 3600
Q_NB = f"""
SELECT h.tic, h.host_id, e.source_id AS nb_id, g.ra AS nb_ra, g.dec AS nb_dec,
       g.phot_g_mean_mag AS nb_g, e.frequency, e.frequency_error, e.reference_time,
       e.derived_primary_ecl_phase, e.derived_primary_ecl_depth,
       e.derived_secondary_ecl_phase, e.model_type, e.global_ranking,
       DISTANCE(POINT('ICRS', h.ra, h.dec), POINT('ICRS', g.ra, g.dec)) * 3600 AS sep_arcsec
FROM tap_upload.hosts AS h
JOIN gaiadr3.gaia_source AS g
  ON 1 = CONTAINS(POINT('ICRS', g.ra, g.dec), CIRCLE('ICRS', h.ra, h.dec, {RADIUS_DEG}))
JOIN gaiadr3.vari_eclipsing_binary AS e ON e.source_id = g.source_id
"""
hp = hosts.dropna(subset=["ra"])[["TICID", "GAIA_DR3", "ra", "dec"]]
parts = []
for i in range(0, len(hp), 500):
    sub = hp.iloc[i:i + 500]
    t = Table({"tic": sub["TICID"].astype("int64").values,
               "host_id": sub["GAIA_DR3"].astype("int64").values,
               "ra": sub["ra"].values, "dec": sub["dec"].values})
    t.write("hosts_upload.xml", format="votable", overwrite=True)
    for attempt in range(3):
        try:
            job = Gaia.launch_job_async(Q_NB, upload_resource="hosts_upload.xml",
                                        upload_table_name="hosts")
            parts.append(job.get_results().to_pandas()); break
        except Exception as err:
            print(f"  batch {i // 500}: {err} - retrying"); time.sleep(20)
    print(f"  done {min(i + 500, len(hp))}/{len(hp)}")
nb = pd.concat(parts, ignore_index=True)
nb = nb[nb["nb_id"] != nb["host_id"]]             # drop the host itself (done in Step 3)
nb.to_csv("data/t16_neighbour_ebs.csv", index=False)
print("Gaia EBs within 60\" of a host (excluding hosts):", len(nb),
      "| hosts with >=1 such neighbour:", nb["tic"].nunique())



INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 500/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 1000/11143


INFO:astroquery:Query finished.


...

INFO: Query finished. [astroquery.utils.tap.core]
  done 10500/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 11000/11143


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]
  done 11143/11143
Gaia EBs within 60" of a host (excluding hosts): 2527 | hosts with >=1 such neighbour: 2305


In [ ]:
# CELL 3 - period + eclipse-time matching (same rules as Step 3)
# ---------------------------------------------------------------------
cand = c[["TICID", "P", "sigP", "Tc_BJD", "sigTc", "dur", "rp/rs", "Tmag"]].rename(columns={"TICID": "tic"})
n2 = nb.merge(cand, on="tic", how="left").merge(
        hosts[["TICID", "phot_g_mean_mag"]].rename(columns={"TICID": "tic", "phot_g_mean_mag": "host_g"}),
        on="tic", how="left")

n2["P_gaia"] = 1 / n2["frequency"]
n2["sigP_gaia"] = n2["frequency_error"] / n2["frequency"] ** 2
n2["T_gaia"] = 2455197.5 + n2["reference_time"] + n2["derived_primary_ecl_phase"] / n2["frequency"]
NSIG, FLOOR = 3.0, 1e-3
n2["k_match"] = np.nan
for k in (0.5, 1.0, 2.0, 3.0, 1/3):              # include 3x aliases (seen in Step 3)
    tol = np.maximum(NSIG * np.hypot(n2["sigP"], k * n2["sigP_gaia"]), FLOOR * n2["P"])
    hit = (np.abs(n2["P"] - k * n2["P_gaia"]) < tol) & n2["k_match"].isna()
    n2.loc[hit, "k_match"] = k
n2["period_match"] = n2["k_match"].notna()

Puse = n2["k_match"] * n2["P_gaia"]
ncyc = np.round((n2["Tc_BJD"] - n2["T_gaia"]) / Puse)
sigT = np.hypot(np.abs(ncyc) * n2["k_match"] * n2["sigP_gaia"], n2["sigTc"])
def offset(T0):
    x = (n2["Tc_BJD"] - T0) / Puse
    return (x - np.round(x)) * Puse
off_p = offset(n2["T_gaia"])
T_sec = n2["T_gaia"] + (n2["derived_secondary_ecl_phase"] - n2["derived_primary_ecl_phase"]) / n2["frequency"]
off_s = offset(T_sec)
n2["ecl_offset_d"] = np.where(n2["derived_secondary_ecl_phase"].notna() & (np.abs(off_s) < np.abs(off_p)),
                              off_s, off_p)
n2["epoch_unconstrained"] = n2["period_match"] & (sigT > 0.25 * Puse)
n2["epoch_match"] = (n2["period_match"] & ~n2["epoch_unconstrained"]
                     & (np.abs(n2["ecl_offset_d"]) < NSIG * sigT + 0.5 * n2["dur"]))



In [ ]:
# CELL 4 - dilution test: can this neighbour produce the T16 depth?
# Depth seen on the blend = (neighbour's own eclipse depth) x (its share of the light)
# G-band is used as a proxy for the TESS band (approximation, stated in the paper).
# ---------------------------------------------------------------------
n2["T16_depth"] = n2["rp/rs"] ** 2                                     # approximate transit depth
n2["flux_ratio"] = 10 ** (-0.4 * (n2["nb_g"] - n2["host_g"]))          # neighbour / host flux
n2["share"] = n2["flux_ratio"] / (1 + n2["flux_ratio"])
n2["nb_depth_frac"] = 1 - 10 ** (-0.4 * n2["derived_primary_ecl_depth"])
n2["diluted_depth"] = n2["nb_depth_frac"] * n2["share"]
n2["can_explain"] = n2["diluted_depth"] >= 0.5 * n2["T16_depth"]       # factor-2 tolerance (band, limb darkening)
n2["max_possible"] = n2["share"] >= n2["T16_depth"]                    # even a 100 % eclipse could do it
n2.to_csv("data/t16_neighbour_flags.csv", index=False)


In [ ]:
# CELL 5 - summary
# ---------------------------------------------------------------------
strong = n2["epoch_match"] & n2["can_explain"]
print("neighbour EBs (pairs):", len(n2), "| distinct hosts:", n2["tic"].nunique())
print("period match:", n2["period_match"].sum(), "| + eclipse time:", n2["epoch_match"].sum(),
      "| + depth possible (STRONG):", strong.sum(), "| distinct T16 candidates:", n2.loc[strong, "tic"].nunique())
print("period match but eclipse time unconstrained:", n2["epoch_unconstrained"].sum())
cols = ["tic", "Tmag", "P", "P_gaia", "k_match", "ecl_offset_d", "sep_arcsec", "host_g", "nb_g",
        "T16_depth", "diluted_depth", "epoch_match", "can_explain"]
n2.loc[n2["period_match"], cols].sort_values(["epoch_match", "can_explain"], ascending=False).head(40)


neighbour EBs (pairs): 2527 | distinct hosts: 2305
period match: 744 | + eclipse time: 690 | + depth possible (STRONG): 687 | distinct T16 candidates: 687
period match but eclipse time unconstrained: 8


,tic,Tmag,P,P_gaia,k_match,ecl_offset_d,sep_arcsec,host_g,nb_g,T16_depth,diluted_depth,epoch_match,can_explain
1,308544025,15.81444,1.460340,1.461031,1.0,0.110522,27.796804,16.370964,18.123775,0.027556,0.035680,True,True
3,55403924,13.78173,1.538600,1.537590,1.0,-0.003885,12.818183,14.241175,18.850115,0.004583,0.010061,True,True
8,155946928,15.35346,2.031400,2.031534,1.0,0.018907,10.075204,16.170956,18.969496,0.011449,0.028419,True,True
12,389565802,13.95516,6.637700,6.636970,1.0,-0.109054,27.124195,14.513579,16.601435,0.012814,0.069730,True,True
14,277100291,15.40275,4.293900,4.295371,1.0,0.020107,32.411558,16.239014,17.492847,0.045796,0.171094,True,True
19,79103716,15.65827,1.703580,3.407265,0.5,0.004307,26.114926,16.076481,17.166891,0.029929,0.137208,True,True
22,300089996,12.96545,1.435870,1.436186,1.0,0.025944,30.995753,13.305511,16.505747,0.006368,0.016628,True,True
23,277101917,14.93677,2.124400,2.124371,1.0,0.000537,32.794804,15.722185,18.496958,0.010609,0.037665,True,True
33,309760488,15.61825,3.941200,7.871912,0.5,-0.013282,25.991622,16.041864,17.897749,0.040000,0.089343,True,True
35,183496329,15.59719,5.123800,5.126914,1.0,-0.014087,18.904682,16.660936,16.827808,0.111556,0.199008,True,True


In [ ]:
# 5a: unique EB sources, shared signals, and overlap with known catalogues
S = n2[n2["epoch_match"] & n2["can_explain"]].copy()
print("STRONG pairs:", len(S), "| T16 candidates:", S["tic"].nunique(),
      "| distinct Gaia EBs responsible:", S["nb_id"].nunique())
shared = S.groupby("nb_id")["tic"].nunique()
print("Gaia EBs contaminating >1 T16 candidate:", (shared > 1).sum(),
      "| T16 candidates involved:", S[S["nb_id"].isin(shared[shared > 1].index)]["tic"].nunique())

# overlap 1: is the T16 candidate already a TOI? (and with which verdict)
S = S.merge(toi[["tid", "toi", "tfopwg_disp"]].rename(columns={"tid": "tic"}), on="tic", how="left")
print(S.drop_duplicates("tic")["tfopwg_disp"].fillna("not a TOI").value_counts())

# overlap 2: is the neighbour EB already in Kostov 2025 or Prsa 2022? (match by position, 5")
from astropy.coordinates import SkyCoord
import astropy.units as u
def pos_match(df, ra_col, dec_col, label):
    cat = SkyCoord(df[ra_col].values * u.deg, df[dec_col].values * u.deg)
    me = SkyCoord(S["nb_ra"].values * u.deg, S["nb_dec"].values * u.deg)
    idx, d2d, _ = me.match_to_catalog_sky(cat)
    S[f"in_{label}"] = d2d.arcsec < 5
    print(f"neighbour EB already in {label}: {S[f'in_{label}'].sum()} of {len(S)}")
pos_match(kos, "RAdeg", "DEdeg", "Kostov2025")
pra = [c_ for c_ in prsa_main.columns if c_.lower().startswith(("ra", "_ra"))][0]
pde = [c_ for c_ in prsa_main.columns if c_.lower().startswith(("de", "_de"))][0]
print("Prsa position columns used:", pra, pde)
pos_match(prsa_main, pra, pde, "Prsa2022")

S["new_flag"] = S["tfopwg_disp"].isna() & ~S["in_Kostov2025"] & ~S["in_Prsa2022"]
print("NEW blend flags (not a TOI, neighbour not in Kostov/Prsa):",
      S.loc[S["new_flag"], "tic"].nunique())
S.to_csv("data/t16_blend_flags.csv", index=False)

STRONG pairs: 687 | T16 candidates: 687 | distinct Gaia EBs responsible: 642
Gaia EBs contaminating >1 T16 candidate: 40 | T16 candidates involved: 85
tfopwg_disp
not a TOI    685
FP             1
CP             1
Name: count, dtype: int64
neighbour EB already in Kostov2025: 0 of 687
Prsa position columns used: RAJ2000 DEJ2000
neighbour EB already in Prsa2022: 0 of 687
NEW blend flags (not a TOI, neighbour not in Kostov/Prsa): 685


In [ ]:
# 6a: positive control - Kostov EBs should match themselves, and some T16 hosts may sit on Kostov EBs
kc = SkyCoord(kos["RAdeg"].values * u.deg, kos["DEdeg"].values * u.deg)
_, d_self, _ = kc[:200].match_to_catalog_sky(kc, nthneighbor=1)
print("control (Kostov vs itself, should be 200):", (d_self.arcsec < 5).sum())
hc = SkyCoord(hosts["ra"].values * u.deg, hosts["dec"].values * u.deg)
_, d_h, _ = hc.match_to_catalog_sky(kc)
print("T16 hosts that ARE Kostov EBs (within 5\"):", (d_h.arcsec < 5).sum())

# 6b: chance rate - shuffle (P, Tc, ...) among T16 candidates, redo the neighbour matching
rng = np.random.default_rng(42)
cand_cols = ["P", "sigP", "Tc_BJD", "sigTc", "dur", "rp/rs"]
pool = c[cand_cols].dropna().to_numpy()
def n_strong(df):
    Pg, sPg = 1 / df["frequency"], df["frequency_error"] / df["frequency"] ** 2
    Tg = 2455197.5 + df["reference_time"] + df["derived_primary_ecl_phase"] / df["frequency"]
    best = np.zeros(len(df), bool)
    for k in (0.5, 1.0, 2.0, 3.0, 1/3):
        tol = np.maximum(3 * np.hypot(df["sigP"], k * sPg), 1e-3 * df["P"])
        pm = np.abs(df["P"] - k * Pg) < tol
        Pu = k * Pg
        x = (df["Tc_BJD"] - Tg) / Pu
        off = np.abs(x - np.round(x)) * Pu
        sT = np.hypot(np.abs(np.round(x)) * k * sPg, df["sigTc"])
        ok = pm & (sT <= 0.25 * Pu) & (off < 3 * sT + 0.5 * df["dur"])
        best |= ok.to_numpy()
    dil = (1 - 10 ** (-0.4 * df["derived_primary_ecl_depth"])) * df["share"] >= 0.5 * df["rp/rs"] ** 2
    return int((best & dil.to_numpy()).sum())
print("real data:", n_strong(n2))
fake = []
for it in range(100):
    d = n2.copy()
    d[cand_cols] = pool[rng.integers(0, len(pool), len(d))]
    fake.append(n_strong(d))
print(f"shuffled: mean {np.mean(fake):.2f}, max {np.max(fake)}, 99th pct {np.percentile(fake, 99):.1f}")

# 6c: which CP is in the blend list?
print(S.loc[S["tfopwg_disp"].isin(["CP", "FP"]), ["tic", "toi", "tfopwg_disp", "P", "P_gaia", "sep_arcsec", "nb_g"]])

control (Kostov vs itself, should be 200): 200
T16 hosts that ARE Kostov EBs (within 5"): 2
real data: 670
shuffled: mean 4.71, max 10, 99th pct 10.0
           tic      toi tfopwg_disp        P    P_gaia  sep_arcsec       nb_g
161  179155220   964.01          FP  0.60281  1.205251   19.169173  14.489795
318  364395234  1232.01          CP  1.37567  1.375745   13.744758  14.723995
